# Annotate public IDR data (OME-Zarr)

The [Image Data Resource (IDR)](https://idr.openmicroscopy.org) publishes many studies as
OME-Zarr on public S3 storage. This notebook annotates such a plate with micro-SAM and
trains a model on the result. No OMERO login is needed and nothing is uploaded: the
training set is kept in a local folder (`storage: local`).

The example is HeLa nuclei (H2B-GFP) from the MitoCheck screen (study idr0013A). Only
studies converted to OME-Zarr can be used; step 1 checks a screen for you.

The calls to the IDR web API in step 1 are based on the IDR's own
[IDR API example notebook](https://github.com/IDR/idr-notebooks/blob/master/IDR_API_example_script.ipynb)
(IDR/idr-notebooks).

The same workflow as a marimo notebook: `notebooks/marimo/idr-zarr-demo.py`.

In [ ]:
from datetime import datetime

import pandas as pd
import requests

from omero_annotate_ai import (
    TrainingSet,
    load_config_from_yaml,
    reorganize_local_data_for_training,
    run_training,
    setup_training,
)

## 1. Find a plate (optional)

Give an IDR screen ID and the study's OME-Zarr folder (study name plus `A`, `B`, … as on
the IDR, e.g. `idr0013A`). The IDR web API lists the screen's plates and each plate is
checked for an OME-Zarr copy.

In [ ]:
screen_id = 1101  # idr0013-neumann-mitocheck/screenA
zarr_study = "idr0013A"
n_plates = 5  # plates to check

plates = requests.get(
    "https://idr.openmicroscopy.org/webclient/api/plates/",
    params={"id": screen_id},
    timeout=30,
).json()["plates"][:n_plates]
found = []
for plate in plates:
    url = (
        f"https://livingobjects.ebi.ac.uk/idr/zarr/v0.4/{zarr_study}/{plate['id']}.zarr"
    )
    has_zarr = requests.head(url + "/.zattrs", timeout=15).status_code == 200
    found.append(
        {
            "plate": plate["id"],
            "name": plate["name"],
            "OME-Zarr": url if has_zarr else "-",
        }
    )
pd.DataFrame(found)

## 2. Choose the images

The defaults come from `notebooks/marimo/configs/idr0013-mitocheck-zarr.yaml`. Change the
values below to use another plate, other wells or more images.

In [ ]:
config = load_config_from_yaml("../../marimo/configs/idr0013-mitocheck-zarr.yaml")
config.name = f"idr_zarr_{datetime.now():%Y%m%d_%H%M%S}"

# --- edit these -------------------------------------------------------------
config.zarr.uris = ["https://livingobjects.ebi.ac.uk/idr/zarr/v0.4/idr0013A/3451.zarr"]
config.zarr.wells = ["A1", "B2", "C3", "D4"]  # None = all wells
config.zarr.level = 0  # resolution level, 0 = full resolution
config.sampling.label_channel = 0  # GFP-tagged histone H2B: nuclei
config.training.train_n = 2
config.training.validate_n = 1
config.split.seed = 42
# ----------------------------------------------------------------------------

## 3. Create the training set

Lists the plate's wells and fields and picks the images (one random frame of each
time-lapse in the example). Only metadata is read here.

In [ ]:
ts = TrainingSet.create(config)
print(f"{len(ts.units())} units in {ts.bundle_dir}")
pd.DataFrame(
    [
        {"split": u.category, "well": u.well, "field": u.field, "frame": u.timepoint}
        for u in ts.units()
    ]
)

## 4. Annotate

napari opens with micro-SAM. Segment the nuclei, then close the window; each image you
segmented is saved. Open the set again later with `TrainingSet.open(ts.bundle_dir)`.

In [ ]:
saved = ts.annotate(note=None)  # e.g. note="nuclei only, small debris skipped"
print(f"{saved} units saved in {ts.bundle_dir}")

## 5. Train

The annotated images are copied into a training layout next to the set's folder and a
micro-SAM model is fine-tuned on them. A few epochs only check that it runs; use 50 or
more for a useful model.

In [ ]:
epochs = 5

if not any(u.processed for u in ts.units()):
    raise ValueError("No annotations yet: run step 4 first.")
data = reorganize_local_data_for_training(ts.config, annotation_dir=ts.bundle_dir)
training = setup_training(
    data,
    model_name=f"idr_microsam_{datetime.now():%Y%m%d_%H%M%S}",
    model_type="vit_b_lm",
    epochs=epochs,
    batch_size=1,
    patch_shape=(512, 512),
)
result = run_training(training, framework="microsam")
print(f"Model: {result.get('final_model_path', '-')}")
print(f"Output: {result['output_dir']}")